<a href="https://colab.research.google.com/" target="_parent">Open in Google Colab</a>

# AI Agent Loop Demo — Student Campus Protocol Dataset

This notebook is a copy of the original AI Agent Loop Demo, adapted to a **new dataset**: a fictional Student Campus Protocol Handbook.

The agent can search the supplied documents, use the matching passages to answer questions, and say when the documents do not contain an answer.

**Dataset:** four student-guideline documents covering attendance, campus conduct, library use, and event participation.

## 1. Setup — Gemini API key

1. Open [Google AI Studio](https://aistudio.google.com/apikey).
2. Create or select an API key.
3. In Google Colab, open the **Secrets** panel (key icon), add a secret named `GOOGLE_API_KEY`, and enable notebook access.
4. Alternatively, the next cell will ask you to paste the key privately.

Do not write your real API key directly into notebook code or upload it to GitHub.

In [ ]:
# Load the Gemini API key securely.
import os
from getpass import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("A Gemini API key is required.")

os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key loaded.")

In [ ]:
# Install the current Google Gen AI SDK.
%pip install -q -U google-genai
print("Installation complete.")

In [ ]:
# @title Student Campus Protocol Dataset

DOCUMENTS = {
    "attendance_and_academics": (
        "Students are expected to attend all scheduled lectures, laboratory sessions, "
        "and tutorials. If a student is absent because of illness or an emergency, "
        "they should inform the class teacher or department office as soon as possible. "
        "Students must submit assignments before the deadline announced by the course instructor. "
        "Requests for an extension should be made before the deadline and should include a valid reason."
    ),

    "campus_conduct": (
        "Students must carry their valid college identity cards while they are on campus. "
        "They should maintain respectful behaviour toward classmates, faculty members, "
        "staff, and visitors. Damage to classroom furniture, laboratory equipment, or other "
        "college property is not permitted. Students should keep classrooms and common areas "
        "clean and place waste in the designated bins."
    ),

    "library_guidelines": (
        "Students must use their college identity card to borrow books from the library. "
        "A borrowed book must be returned on or before its due date. Students should maintain "
        "silence in reading areas and keep mobile phones on silent mode. Reference books and "
        "journals must be used inside the library unless the librarian grants permission otherwise."
    ),

    "clubs_and_campus_events": (
        "Students who want to organize a club activity or campus event must obtain approval "
        "from the faculty coordinator. The event proposal should mention its purpose, date, "
        "venue, expected number of participants, and any equipment required. Organizers are "
        "responsible for keeping the venue clean after the event. Participants must follow "
        "the instructions given by event volunteers and college staff."
    )
}

# Check that the dataset contains at least four documents.
assert len(DOCUMENTS) >= 4
print(f"Loaded {len(DOCUMENTS)} documents.")
for name in DOCUMENTS:
    print("-", name)

In [ ]:
# @title Define the document-search tool

def search_documents(query: str) -> str:
    """Return the two document passages most relevant to a query."""
    query_words = set(query.lower().split())
    results = []

    for doc_name, content in DOCUMENTS.items():
        content_lower = content.lower()
        score = sum(1 for word in query_words if len(word) > 2 and word in content_lower)
        if score > 0:
            results.append((score, doc_name, content))

    results.sort(key=lambda item: item[0], reverse=True)

    if not results:
        return "No relevant documents found for that query."

    return "\\n\\n".join(
        f"[Source: {name}]\\n{content}"
        for _, name, content in results[:2]
    )

# Quick search test.
print(search_documents("identity card library"))

In [ ]:
# @title Configure Gemini and declare the search tool

from google import genai
from google.genai import types
import time

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

MODEL_NAME = "gemini-2.5-flash"

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the student campus protocol documents for relevant information.",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "The topic or question to search for."
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    system_instruction=(
        "You are a student support assistant. Answer questions using only the "
        "provided Student Campus Protocol documents. When information is needed, "
        "call the search_documents tool. Do not invent rules or add outside facts. "
        "If the documents do not contain the answer, clearly say that the information "
        "is not available in the supplied documents."
    )
)

print("Model and document-search tool are ready:", MODEL_NAME)

In [ ]:
# @title Run the agent loop

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )
        except Exception as error:
            if "429" in str(error) and attempt < max_retries - 1:
                wait_seconds = 15 * (attempt + 1)
                print(f"Rate limit reached. Waiting {wait_seconds} seconds...")
                time.sleep(wait_seconds)
            else:
                raise

def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\\n" + "-" * 60)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Keep the complete model response in the conversation.
        contents.append(model_content)

        function_calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        if not function_calls:
            answer = response.text or "The model did not return a text answer."
            print("\\nFinal answer:\\n" + answer)
            return answer

        function_results = []
        for function_call in function_calls:
            arguments = dict(function_call.args)
            print(f"Step {step}: calling {function_call.name}({arguments})")

            if function_call.name == "search_documents":
                result = search_documents(**arguments)
            else:
                result = "Unknown tool requested."

            print("Search result preview:", result[:180].replace("\\n", " "), "...")

            function_results.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(role="user", parts=function_results)
        )

    print("Agent stopped because the maximum number of steps was reached.")
    return "(Stopped: hit max_steps)"

# First test question.
run_agent("What should students do before organizing a campus event?")

In [ ]:
# @title Ask follow-up questions

questions = [
    "What are the rules for borrowing and returning library books?",
    "What details must be included in a campus event proposal?",
    "What should a student do if they are absent because of illness?",
    "What behaviour is expected from students in classrooms and common areas?",
    # This question is intentionally not answered by the dataset.
    "How many ghosts eat carrots every day?",
    # This asks for general knowledge not contained in the documents.
    "Which is the most populated country in the world?"
]

for index, question in enumerate(questions):
    if index > 0:
        time.sleep(2)  # Small pause between requests.
    print("\\n" + "=" * 70)
    run_agent(question)

In [ ]:
# @title Final checks

assert len(DOCUMENTS) >= 4, "Add at least four documents."
assert all(isinstance(text, str) and len(text.split(". ")) >= 3
           for text in DOCUMENTS.values()), "Each document should contain at least four sentences."
assert len(questions) >= 4, "Add at least four follow-up questions."
assert all("Add a question here." not in question for question in questions)

print("All notebook checks passed.")
print(f"Documents: {len(DOCUMENTS)}")
print(f"Follow-up questions: {len(questions)}")